# TRM Latent-History Sudoku Study

Reproducible Colab/T4 workflow for B0/B1/B2/B3/P1. The 55-minute setting is a runtime cap, not a guarantee of convergence or completion. Scaled-model inference is useful for iteration but is not publication-scale evidence.

In [ ]:
# Setup: Runtime → T4 GPU. Clone the study branch (main does not have it).
import os, pathlib

REPO_URL = os.environ.get(
    "TRM_REPO_URL",
    "https://github.com/Iliyabr/trm-latent-history-attention.git",
)
BRANCH = os.environ.get("TRM_BRANCH", "feature/latent-history-attention")

if pathlib.Path("/content").exists():
    %cd /content
if not pathlib.Path("trm-latent-history-attention").exists():
    !git clone -b "$BRANCH" "$REPO_URL" trm-latent-history-attention
%cd trm-latent-history-attention
!git fetch origin
!git checkout "$BRANCH"
!git pull --ff-only origin "$BRANCH"

# Colab already has torch. requirements.txt pulls adam-atan2/triton, which
# fail metadata generation on Colab Python 3.13.
!python -m pip install -q -r requirements-colab.txt
print("ready", pathlib.Path.cwd())

## Build deterministic data

This creates 900 train bases with 64 augmentations each, 100 unaugmented dev puzzles, and a bounded 1,000-example official test split. The builder writes hashes and asserts leakage before augmentation.

In [ ]:
!python dataset/build_sudoku_baseline_v2.py
import json
manifest = json.load(open("artifacts/data/sudoku_study_v1_manifest.json"))
print(manifest["counts"])
print(manifest["leakage_assertions"])

## Run one job or the suite

Start with dry-run. Change `VARIANT` and `SEED` for one of the 15 jobs. The full suite is serial and may take roughly 15 hours at the one-hour target.

In [ ]:
VARIANT, SEED = "P1", 0
!python experiments/run_study.py single --variant $VARIANT --seed $SEED --dry-run
# Remove --dry-run to train:
# !python experiments/run_study.py single --variant $VARIANT --seed $SEED
# Full serial suite:
# !python experiments/run_study.py suite

## Resume a capped/interrupted run

In [ ]:
# Auto-selects runtime_cap.pt, otherwise the latest complete step checkpoint.
!python experiments/run_study.py resume --variant $VARIANT --seed $SEED --dry-run
# Remove --dry-run after confirming the command.

## Inspect outputs

`metrics.jsonl` includes train throughput/VRAM/runtime, dev metrics, best-checkpoint decisions, and plateau evidence.

In [ ]:
from pathlib import Path
run_dir = Path(f"outputs/study/colab/{VARIANT}-seed{SEED}")
print(sorted(p.name for p in run_dir.glob("*.pt")))
metrics_file = run_dir / "metrics.jsonl"
if metrics_file.exists():
    records = [json.loads(line) for line in metrics_file.read_text().splitlines()]
    print(json.dumps(records[-1], indent=2))
else:
    print("No metrics yet; run training first.")

# After several seeds exist, extract paper tables/figures:
# !python experiments/evaluate_study.py --config config/experiment/sudoku_study_colab.yaml --checkpoint {VARIANT}={run_dir}/best_dev.pt --data data/sudoku-study-v1 --split test --interventions --seed {SEED}
# !python experiments/analyze_results.py --input results/study --output results/study/analysis